In [44]:
#Started 14/8/2025 - Pokemon Worlds 2025 In Anaheim Started 1 Day later
import sys
import random
from typing import Optional, Union

from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import DoublesEnv
from poke_env.environment.env import _EnvPlayer
from poke_env.battle import AbstractBattle, Battle
from poke_env.battle.double_battle import DoubleBattle
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player, RandomPlayer


import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from typing import Any, Dict, Optional

from gymnasium.spaces import Box, Discrete, Space, MultiDiscrete
import torch
import torch.nn as nn
from torch import multiprocessing
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

from ray.rllib.algorithms import PPOConfig
from ray.rllib.core import Columns
from ray.rllib.core.rl_module import RLModuleSpec
from ray.rllib.core.rl_module.apis.value_function_api import ValueFunctionAPI
from ray.rllib.core.rl_module.torch import TorchRLModule
from ray.rllib.env import ParallelPettingZooEnv
from ray.tune.registry import register_env

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (Compose, DoubleToFloat, ObservationNorm, StepCounter,
                          TransformedEnv)


#Custom Env pytorch tutorial
from typing import Optional
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import step_mdp
#---------------------------
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

In [2]:
"""
cd "C:\Austin\Self_Projects\Pokemon_Sim\pokemon-showdown"
node pokemon-showdown start --no-security
"""

'\ncd "C:\\Austin\\Self_Projects\\Pokemon_Sim\\pokemon-showdown"\nnode pokemon-showdown start --no-security\n'

In [ ]:
class VgcEnv(DoublesEnv, EnvBase):

    def __init__(
        self,
        seed=None,
        #EnvBase Variables
        device="cpu",
        batch_size = [],
        #DoublesEnv Variables
        account_configuration1: Optional[AccountConfiguration] = None,
        account_configuration2: Optional[AccountConfiguration] = None,
        battle_format: str = "gen8randombattle",
        accept_open_team_sheet: Optional[bool] = True,
        start_timer_on_battle_start: bool = True,
        #Custom Input
        team1: Optional[Union[str, Teambuilder]] = None,
        team2: Optional[Union[str, Teambuilder]] = None,
        strict = True,
    ):
        #self.observation_spaces = {
        #    agent: Box(np.array([0]), np.array([1]), dtype=np.int64)
        #    for agent in self.possible_agents
        #}

        DoublesEnv.__init__(
            self,
            #DoublesEnv Variables
            account_configuration1=account_configuration1,
            account_configuration2=account_configuration2,
            battle_format=battle_format,
            accept_open_team_sheet=accept_open_team_sheet,
            start_timer_on_battle_start=start_timer_on_battle_start,
            strict=strict,
        )
        EnvBase.__init__(
            self,
            #EnvBase Variables
            device=device, 
            batch_size=batch_size
        )

        self.agent1.teampreview = self.teampreview
        self.agent2.teampreview = self.teampreview

        self.agent1.update_team(team1)
        self.agent2.update_team(team2)

        if seed is None:
            seed = torch.empty((), dtype=torch.int64).random_().item()
        self.set_seed(seed)

    #EnvBase Required Functions:

    def _reset(self, tensordict):
        print("In Reset")

        obs, _ = self.reset()
        print("observation")
        print(obs)
        out = TensorDict(
            {
                "observations":obs
            },
            batch_size=tensordict.shape,
        )

        print("Out Reset")
        #print(out)

        return out
    
    def _step(self, tensordict):

        print("In Step")
        #print(tensordict)
        
        #observations, reward, terminated, truncated, _ = 
        print(self.step(tensordict["actions"]))

        out = TensorDict(
            {"""
                "observations":observations,
                "reward":reward,
                "terminated":terminated,
                "truncated":truncated,
            """},
            batch_size=tensordict.shape,
        )

        print("Out Step")
        #print(out)

        return out
        

    def _set_seed(self, seed: Optional[int]): #Will write a custom reward function as required
        rng = torch.manual_seed(seed)
        self.rng = rng

    #DoublesEnv Required Functions:

    def calc_reward(self, battle) -> float:
        #Initially using built in reward helper function provided by Poke-Env
        #For simplicity of initiall implementation and testing

        return self.reward_computing_helper(
            battle, fainted_value=2.0, hp_value=1.0, victory_value=30.0
        )


    def embed_battle(self, battle: AbstractBattle): #Will write a custom reward function as required - __NEED TO UPDATE FOR DOUBLE BATTLE__
        assert isinstance(battle, DoubleBattle)
        # -1 indicates that the move does not have a base power
        # or is not available
        moves_base_power = -np.ones(4)
        moves_dmg_multiplier = np.ones(4)
        print(battle.available_moves)
        for mon in battle.available_moves:
            for i, move in enumerate(mon):
                moves_base_power[i] = (
                    move.base_power / 100
                )  # Simple rescaling to facilitate learning
                if battle.opponent_active_pokemon is not None:
                    moves_dmg_multiplier[i] = move.type.damage_multiplier(
                        battle.opponent_active_pokemon.type_1,
                        battle.opponent_active_pokemon.type_2,
                        type_chart=battle.opponent_active_pokemon._data.type_chart,
                    )

        # We count how many pokemons have fainted in each team
        fainted_mon_team = len([mon for mon in battle.team.values() if mon.fainted]) / 6
        fainted_mon_opponent = (
            len([mon for mon in battle.opponent_team.values() if mon.fainted]) / 6
        )

        # Final vector with 10 components
        final_vector = np.concatenate(
            [
                moves_base_power,
                moves_dmg_multiplier,
                [fainted_mon_team, fainted_mon_opponent],
            ]
        )
        return np.float32(final_vector)

        
    
    def teampreview(self, battle: AbstractBattle) -> str: #Will write a custom reward function as required
        members = list(range(1, 7))
        random.shuffle(members)
        return "/team " + "".join([str(c) for c in members[:4]])

    #Helper Functions:

    def print_teams(self):
        print(self.agent1._team.yield_team())
        print(self.agent2._team.yield_team())

    def print_torchrl_env_stats(self):
        print(self.device)
        print(self.batch_size)

    def select_game_team(self):
        pass

    def set_battle(self):
        self.battle1._finished=False
        self.battle2._finished=False

In [111]:
teams =[ """
Typhlosion-Hisui @ Charcoal  
Ability: Blaze  
Level: 50  
Tera Type: Fire  
EVs: 252 SpA / 4 SpD / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Eruption  
- Heat Wave  
- Shadow Ball  
- Protect  

Whimsicott @ Covert Cloak  
Ability: Prankster  
Level: 50  
Tera Type: Dark  
EVs: 244 HP / 4 Def / 4 SpA / 4 SpD / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Tailwind  
- Moonblast  
- Sunny Day  
- Encore  

Ursaluna-Bloodmoon @ Life Orb  
Ability: Mind's Eye  
Level: 50  
Tera Type: Normal  
EVs: 4 HP / 252 SpA / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Hyper Voice  
- Blood Moon  
- Earth Power  
- Protect  

Primarina @ Throat Spray  
Ability: Liquid Voice  
Level: 50  
Tera Type: Grass  
EVs: 244 HP / 52 Def / 108 SpA / 28 SpD / 76 Spe  
Modest Nature  
IVs: 0 Atk  
- Moonblast  
- Hyper Voice  
- Haze  
- Protect  

Meowscarada @ Focus Sash  
Ability: Protean  
Level: 50  
Tera Type: Grass  
EVs: 4 HP / 252 Atk / 252 Spe  
Jolly Nature  
- Flower Trick  
- Knock Off  
- U-turn  
- Protect  

Farigiraf @ Safety Goggles  
Ability: Armor Tail  
Level: 50  
Tera Type: Fire  
EVs: 228 HP / 156 Def / 124 SpD  
Relaxed Nature  
IVs: 0 Atk / 0 Spe  
- Psychic Noise  
- Hyper Voice  
- Helping Hand  
- Trick Room  
""",

"""
Annihilape @ Lum Berry  
Ability: Defiant  
Level: 50  
Tera Type: Water  
EVs: 180 HP / 36 Atk / 12 Def / 28 SpD / 252 Spe  
Adamant Nature  
- Rage Fist  
- Drain Punch  
- Bulk Up  
- Protect  

Maushold @ Safety Goggles  
Ability: Friend Guard  
Level: 50  
Tera Type: Ghost  
EVs: 252 HP / 4 Def / 252 Spe  
Timid Nature  
- Follow Me  
- Beat Up  
- Taunt  
- Protect  

Sinistcha @ Sitrus Berry  
Ability: Hospitality  
Level: 50  
Tera Type: Fairy  
EVs: 236 HP / 36 Def / 236 SpD  
Sassy Nature  
IVs: 0 Atk / 0 Spe  
- Matcha Gotcha  
- Life Dew  
- Trick Room  
- Rage Powder  

Archaludon @ Assault Vest  
Ability: Stamina  
Level: 50  
Tera Type: Grass  
EVs: 212 HP / 12 Def / 44 SpA / 212 SpD / 28 Spe  
Modest Nature  
- Electro Shot  
- Flash Cannon  
- Body Press  
- Draco Meteor  

Pelipper @ Focus Sash  
Ability: Drizzle  
Level: 50  
Tera Type: Stellar  
EVs: 252 SpA / 4 SpD / 252 Spe  
Modest Nature  
- Hurricane  
- Weather Ball  
- Wide Guard  
- Protect  

Hydreigon @ Scope Lens  
Ability: Levitate  
Level: 50  
Shiny: Yes  
Tera Type: Steel  
EVs: 252 SpA / 4 SpD / 252 Spe  
Timid Nature  
- Draco Meteor  
- Dark Pulse  
- Focus Energy  
- Protect  

"""
]

In [131]:
vgc_format="gen9vgc2025regh"
env = VgcEnv(battle_format=vgc_format ,team1 = teams[0], team2 = teams[1], strict = False)

In [54]:
env.print_teams()

Typhlosion-Hisui||charcoal|blaze|eruption,heatwave,shadowball,protect|Timid|,,,252,4,252||,0,,,,||50|,,,,,Fire]Whimsicott||covertcloak|prankster|tailwind,moonblast,sunnyday,encore|Timid|244,,4,4,4,252||,0,,,,||50|,,,,,Dark]Ursaluna-Bloodmoon||lifeorb|mindseye|hypervoice,bloodmoon,earthpower,protect|Timid|4,,,252,,252||,0,,,,||50|,,,,,Normal]Primarina||throatspray|liquidvoice|moonblast,hypervoice,haze,protect|Modest|244,,52,108,28,76||,0,,,,||50|,,,,,Grass]Meowscarada||focussash|protean|flowertrick,knockoff,uturn,protect|Jolly|4,252,,,,252||||50|,,,,,Grass]Farigiraf||safetygoggles|armortail|psychicnoise,hypervoice,helpinghand,trickroom|Relaxed|228,,156,,124,||,0,,,,0||50|,,,,,Fire
Annihilape||lumberry|defiant|ragefist,drainpunch,bulkup,protect|Adamant|180,36,12,,28,252||||50|,,,,,Water]Maushold||safetygoggles|friendguard|followme,beatup,taunt,protect|Timid|252,,4,,,252||||50|,,,,,Ghost]Sinistcha||sitrusberry|hospitality|matchagotcha,lifedew,trickroom,ragepowder|Sassy|236,,36,,236,||,0,,

In [132]:
td = TensorDict({},[],)

#env.agent1.battle_against(env.agent2)

td = env._reset(td)

print(env.agent1.teampreview(env.battle1))
print(env.agent2.teampreview(env.battle1))
print(env.battle1.finished)

In Reset
None
None
[[psychicnoise (Move object), hypervoice (Move object), helpinghand (Move object), trickroom (Move object)], [moonblast (Move object), hypervoice (Move object), haze (Move object), protect (Move object)]]


AttributeError: 'list' object has no attribute 'type_1'

In [117]:
td = TensorDict({
        "actions":{
                str(name): (
                    env.action_space(name).sample()
                )
                for name in env.agents
            }})

#print(td["actions","VgcEnv aqul9"])
#print(td["actions","VgcEnv tadap"])

out = env._step(td)

In Step


AssertionError: 

In [101]:
out

TensorDict(
    fields={
        observations: TensorDict(
            fields={
                VgcEnv aqul9: NonTensorData(data=None, batch_size=torch.Size([]), device=None),
                VgcEnv tadap: NonTensorData(data=None, batch_size=torch.Size([]), device=None)},
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        reward: TensorDict(
            fields={
                VgcEnv aqul9: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False),
                VgcEnv tadap: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False)},
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        terminated: TensorDict(
            fields={
                VgcEnv aqul9: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.bool, is_shared=False),
                VgcEnv tadap: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.bool, is_shared=False)},
 

In [15]:
battle = DoubleBattle("a", env.agent1.username, None, 9)

In [72]:
#env.agent1.battle_against(env.agent2)

print(env.agent1.battle_queue.empty())
print(env.agent2.battle_queue.empty())
print(env.agent1.battle)
print(env.agent2.battle)
print(env.battle1)
print(env.battle2)

False
False
None
None


In [27]:
actions = TensorDict({
                str(env.agent1.name): env.order_to_action(Player.choose_random_move(battle), battle),
                for name, battle in zip(env.agents, [env.battle1, env.battle2])
            },
            [],)

SyntaxError: invalid syntax (4062672855.py, line 3)

In [26]:
print(env.battle1)

None


In [15]:
env.action_spaces

{'VgcEnv rncy2': MultiDiscrete([107 107]),
 'VgcEnv 5l6bt': MultiDiscrete([107 107])}